# 🔍 Notebook 1 — Exploration du jeu de données d'emails de phishing

Dans le notebook précédent, tu as appris ce qu'est le phishing et tu as
regardé quelques exemples à la main. Maintenant, nous allons **explorer le
jeu de données complet** avec du code et des visualisations pour découvrir
des motifs (patterns) qui pourraient aider un modèle de machine learning
à distinguer les emails sûrs du phishing.

---

## Ce que tu vas apprendre dans ce notebook

1. Si les deux classes sont équilibrées ou non.
2. Si les emails de phishing sont plus courts ou plus longs que les emails sûrs.
3. Quels mots apparaissent le plus souvent dans chaque classe.
4. Comment faire un nettoyage basique du texte (tokenisation).
5. Comment repérer des échantillons bruités ou mal étiquetés.

À la fin, tu devrais pouvoir répondre à des questions comme :
- *\"Les emails de phishing sont-ils plus courts ?\"*
- *\"Quels mots sont typiques du phishing ?\"*
- *\"Y a-t-il des emails qui semblent mal étiquetés ?\"*

---

## 0 — Mise en place

Importons les bibliothèques nécessaires et chargeons les données.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import re
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize": (10, 5),
    "font.size": 13,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

COLORS = {"safe": "#2ecc71", "phishing": "#e74c3c"}

df = pd.read_csv("data/emails_clean.csv")
print(f"{len(df):,} emails charg\u00e9s.")
df.head(3)

---

## 1 — Équilibre des classes

La première chose à vérifier dans tout problème de classification est :
**combien d'exemples avons-nous pour chaque classe ?**

Si une classe est beaucoup plus grande que l'autre, le modèle pourrait
\"tricher\" en prédisant toujours la classe majoritaire et obtenir quand même
une bonne précision.

Par exemple, si 95% des emails sont sûrs, un modèle qui **dit toujours \"sûr\"**
aurait 95% de précision — mais il raterait chaque email de phishing !

In [ ]:
counts = df["label"].value_counts().sort_index()
label_names = ["S\u00fbr", "Phishing"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bars = axes[0].bar(label_names, counts.values,
                   color=[COLORS["safe"], COLORS["phishing"]],
                   edgecolor="white", linewidth=1.5)
for bar, count in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f"{count:,}", ha="center", fontweight="bold", fontsize=14)
axes[0].set_ylabel("Nombre d'emails")
axes[0].set_title("Distribution des classes")
axes[0].set_ylim(0, counts.max() * 1.15)

axes[1].pie(counts.values, labels=label_names,
            colors=[COLORS["safe"], COLORS["phishing"]],
            autopct="%1.1f%%", startangle=90,
            textprops={"fontsize": 14})
axes[1].set_title("Proportions des classes")

plt.tight_layout()
plt.show()

ratio = counts.min() / counts.max()
print(f"Ratio d'\u00e9quilibre (minorit\u00e9 / majorit\u00e9) : {ratio:.2f}")
if ratio > 0.8:
    print("\u2192 Le jeu de donn\u00e9es est bien \u00e9quilibr\u00e9 !")
elif ratio > 0.5:
    print("\u2192 L\u00e9g\u00e8rement d\u00e9s\u00e9quilibr\u00e9, mais g\u00e9rable.")
else:
    print("\u2192 Le jeu de donn\u00e9es est d\u00e9s\u00e9quilibr\u00e9 \u2014 il faudra en tenir compte.")

### Question pour toi

D'après le graphique ci-dessus :
- Le jeu de données est-il équilibré ou déséquilibré ?
- Si un modèle prédisait toujours \"sûr\", quelle précision obtiendrait-il ?

---

## 2 — Longueur des emails

Les emails de phishing sont-ils plus courts ou plus longs que les emails sûrs ?
Découvrons-le.

On va mesurer la longueur de deux façons :
- **Nombre de caractères** (longueur totale du texte)
- **Nombre de mots** (en découpant sur les espaces)

In [ ]:
df["num_chars"] = df["text"].str.len()
df["num_words"] = df["text"].str.split().str.len()

for label, name in [(0, "S\u00fbr"), (1, "Phishing")]:
    subset = df[df["label"] == label]
    print(f"\nEmails {name} :")
    print(f"  Caract\u00e8res \u2014 moyenne : {subset['num_chars'].mean():.0f}, "
          f"m\u00e9diane : {subset['num_chars'].median():.0f}")
    print(f"  Mots       \u2014 moyenne : {subset['num_words'].mean():.0f}, "
          f"m\u00e9diane : {subset['num_words'].median():.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

safe = df[df["label"] == 0]
phish = df[df["label"] == 1]

# On plafonne au 95e percentile pour une vue plus propre
cap_chars = int(df["num_chars"].quantile(0.95))
axes[0].hist(safe["num_chars"].clip(upper=cap_chars), bins=50,
             alpha=0.6, color=COLORS["safe"], label="S\u00fbr", density=True)
axes[0].hist(phish["num_chars"].clip(upper=cap_chars), bins=50,
             alpha=0.6, color=COLORS["phishing"], label="Phishing", density=True)
axes[0].set_xlabel("Nombre de caract\u00e8res")
axes[0].set_ylabel("Densit\u00e9")
axes[0].set_title("Longueur des emails (caract\u00e8res)")
axes[0].legend()

cap_words = int(df["num_words"].quantile(0.95))
axes[1].hist(safe["num_words"].clip(upper=cap_words), bins=50,
             alpha=0.6, color=COLORS["safe"], label="S\u00fbr", density=True)
axes[1].hist(phish["num_words"].clip(upper=cap_words), bins=50,
             alpha=0.6, color=COLORS["phishing"], label="Phishing", density=True)
axes[1].set_xlabel("Nombre de mots")
axes[1].set_ylabel("Densit\u00e9")
axes[1].set_title("Longueur des emails (mots)")
axes[1].legend()

plt.tight_layout()
plt.show()

### Boîtes à moustaches pour une autre vue

Les boîtes à moustaches (box plots) montrent la **médiane** (trait au milieu),
l'**écart interquartile** (la boîte, couvrant du 25e au 75e percentile), et les
**valeurs aberrantes** (points).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

data_chars = [safe["num_chars"].clip(upper=cap_chars),
              phish["num_chars"].clip(upper=cap_chars)]
bp1 = axes[0].boxplot(data_chars, labels=["S\u00fbr", "Phishing"], patch_artist=True)
for patch, color in zip(bp1["boxes"], [COLORS["safe"], COLORS["phishing"]]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[0].set_ylabel("Nombre de caract\u00e8res")
axes[0].set_title("Longueur en caract\u00e8res par classe")

data_words = [safe["num_words"].clip(upper=cap_words),
              phish["num_words"].clip(upper=cap_words)]
bp2 = axes[1].boxplot(data_words, labels=["S\u00fbr", "Phishing"], patch_artist=True)
for patch, color in zip(bp2["boxes"], [COLORS["safe"], COLORS["phishing"]]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_ylabel("Nombre de mots")
axes[1].set_title("Nombre de mots par classe")

plt.tight_layout()
plt.show()

### Question pour toi

- Les emails de phishing sont-ils généralement plus courts ou plus longs que les emails sûrs ?
- La longueur de l'email à elle seule pourrait-elle être un bon critère de classification ? Pourquoi ou pourquoi pas ?

---

## 3 — Nettoyage du texte (tokenisation)

Avant de pouvoir compter les mots, il faut **nettoyer le texte**. Les emails bruts
contiennent des majuscules, de la ponctuation, des chiffres et des caractères spéciaux
qui ne nous aident pas à distinguer le phishing du sûr.

La **tokenisation** est le processus qui consiste à :
1. Tout convertir en minuscules
2. Supprimer la ponctuation et les caractères spéciaux
3. Découper en mots individuels (appelés *tokens*)
4. Supprimer les mots très courants qui ne portent pas de sens (*stop words*)
   comme \"the\", \"is\", \"and\", \"to\", etc.

Voyons un exemple :

> **Note :** Comme notre jeu de données est en anglais, les *stop words* sont également en anglais.

In [ ]:
# --- Stop words (mots vides) ---
# Ce sont des mots anglais très courants qui n'aident pas à distinguer les classes.

STOP_WORDS = set("""
a about above after again against all am an and any are aren't as at be because
been before being below between both but by can can't cannot could couldn't did
didn't do does doesn't doing don't down during each few for from further get got
had hadn't has hasn't have haven't having he he'd he'll he's her here here's hers
herself him himself his how how's i i'd i'll i'm i've if in into is isn't it it's
its itself let's me more most mustn't my myself no nor not of off on once only or
other ought our ours ourselves out over own same shan't she she'd she'll she's
should shouldn't so some such than that that's the their theirs them themselves
then there there's these they they'd they'll they're they've this those through to
too under until up us very was wasn't we we'd we'll we're we've were weren't what
what's when when's where where's which while who who's whom why why's will with
won't would wouldn't you you'd you'll you're you've your yours yourself yourselves
""".split())

def clean_text(text):
    """Nettoie un email : minuscules, suppression des caract\u00e8res sp\u00e9ciaux, tokenisation, suppression des stop words."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return tokens

sample = "URGENT!! Click here to verify your PayPal account: http://evil-link.com/steal"
print("Original :")
print(f"  {sample}")
print("\nApr\u00e8s nettoyage :")
print(f"  {clean_text(sample)}")

In [ ]:
df["tokens"] = df["text"].apply(clean_text)
df["num_tokens"] = df["tokens"].apply(len)

print("Les 3 premiers emails apr\u00e8s tokenisation :\n")
for i, row in df.head(3).iterrows():
    tag = "PHISHING" if row["label"] == 1 else "S\u00dbR"
    print(f"[{tag}] {row['tokens'][:15]}...")
    print()

---

## 4 — Mots les plus fréquents par classe

Comptons maintenant quels mots apparaissent le plus souvent dans les emails
**sûrs** vs. les emails de **phishing**. Si certains mots sont beaucoup plus
fréquents dans une classe, un modèle pourrait les utiliser comme indices.

In [ ]:
def get_top_words(dataframe, label, n=20):
    """Compte les mots les plus fr\u00e9quents pour un label donn\u00e9."""
    subset = dataframe[dataframe["label"] == label]
    all_tokens = [token for tokens in subset["tokens"] for token in tokens]
    return Counter(all_tokens).most_common(n)

top_safe = get_top_words(df, label=0, n=20)
top_phish = get_top_words(df, label=1, n=20)

print("Top 20 des mots dans les emails S\u00dbRS :")
for word, count in top_safe:
    print(f"  {word:20s} {count:>6,}")

print("\nTop 20 des mots dans les emails de PHISHING :")
for word, count in top_phish:
    print(f"  {word:20s} {count:>6,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

words_s, counts_s = zip(*reversed(top_safe))
axes[0].barh(words_s, counts_s, color=COLORS["safe"], edgecolor="white")
axes[0].set_title("Top 20 des mots \u2014 Emails s\u00fbrs", fontsize=14)
axes[0].set_xlabel("Nombre d'occurrences")

words_p, counts_p = zip(*reversed(top_phish))
axes[1].barh(words_p, counts_p, color=COLORS["phishing"], edgecolor="white")
axes[1].set_title("Top 20 des mots \u2014 Emails phishing", fontsize=14)
axes[1].set_xlabel("Nombre d'occurrences")

plt.tight_layout()
plt.show()

### Mots *distinctifs* de chaque classe

Les listes de mots fréquents ci-dessus partagent probablement beaucoup de
mots communs. Ce qui nous intéresse vraiment, ce sont les mots qui
apparaissent **beaucoup plus dans une classe que dans l'autre**.

Ci-dessous, on calcule un ratio simple : pour chaque mot, on regarde à quel
point il est plus fréquent dans le phishing vs. le sûr (et inversement).

In [ ]:
safe_tokens = [t for tokens in df[df["label"]==0]["tokens"] for t in tokens]
phish_tokens = [t for tokens in df[df["label"]==1]["tokens"] for t in tokens]

safe_freq = Counter(safe_tokens)
phish_freq = Counter(phish_tokens)

total_safe = sum(safe_freq.values())
total_phish = sum(phish_freq.values())

all_words = set(safe_freq.keys()) | set(phish_freq.keys())

MIN_COUNT = 30
ratios = {}
for word in all_words:
    sc = safe_freq.get(word, 0)
    pc = phish_freq.get(word, 0)
    if sc + pc < MIN_COUNT:
        continue
    safe_rate = sc / total_safe
    phish_rate = pc / total_phish
    ratios[word] = phish_rate / (safe_rate + 1e-10)

most_phishy = sorted(ratios.items(), key=lambda x: x[1], reverse=True)[:15]
most_safe = sorted(ratios.items(), key=lambda x: x[1])[:15]

print("Mots les plus indicatifs de PHISHING :")
for word, ratio in most_phishy:
    print(f"  {word:20s}  {ratio:>6.1f}x plus fr\u00e9quent dans le phishing")

print("\nMots les plus indicatifs de S\u00dbR :")
for word, ratio in most_safe:
    r = 1 / (ratio + 1e-10)
    print(f"  {word:20s}  {r:>6.1f}x plus fr\u00e9quent dans le s\u00fbr")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

words_ph, ratios_ph = zip(*reversed(most_phishy))
axes[0].barh(words_ph, ratios_ph, color=COLORS["phishing"], edgecolor="white")
axes[0].set_title("Mots les plus indicatifs de phishing", fontsize=14)
axes[0].set_xlabel("Ratio de fr\u00e9quence (phishing / s\u00fbr)")

words_sf, ratios_sf = zip(*reversed(most_safe))
safe_ratios_display = [1/(r+1e-10) for r in ratios_sf]
axes[1].barh(words_sf, safe_ratios_display, color=COLORS["safe"], edgecolor="white")
axes[1].set_title("Mots les plus indicatifs de s\u00fbr", fontsize=14)
axes[1].set_xlabel("Ratio de fr\u00e9quence (s\u00fbr / phishing)")

plt.tight_layout()
plt.show()

### Question pour toi

- Les \"mots de phishing\" correspondent-ils aux signaux d'alerte que tu as appris dans le Notebook 0 ?
- Peux-tu imaginer un email sûr qui contiendrait accidentellement des \"mots de phishing\" ?

---

## 5 — Motifs spéciaux

Au-delà des mots individuels, les emails de phishing contiennent souvent des
**motifs** spécifiques : des URLs, des adresses email, des mentions d'argent,
des mots d'urgence, etc.

Vérifions-en quelques-uns.

In [ ]:
def count_pattern(text, pattern):
    """Compte combien de fois un motif regex appara\u00eet dans le texte."""
    return len(re.findall(pattern, text, re.IGNORECASE))

df["has_url"] = df["text"].apply(lambda x: bool(re.search(r"https?://|www\.", x, re.IGNORECASE)))
df["has_urgency"] = df["text"].apply(lambda x: bool(re.search(
    r"urgent|immediately|right away|act now|expires|suspended|limited time|hurry",
    x, re.IGNORECASE)))
df["has_money"] = df["text"].apply(lambda x: bool(re.search(
    r"\$|dollar|money|payment|credit card|bank account|wire transfer|bitcoin",
    x, re.IGNORECASE)))
df["has_click"] = df["text"].apply(lambda x: bool(re.search(
    r"click here|click below|click the link|click this",
    x, re.IGNORECASE)))

features = ["has_url", "has_urgency", "has_money", "has_click"]
feature_names = ["Contient URL", "Mots d'urgence", "Li\u00e9 \u00e0 l'argent", "'Click here'"]

print(f"{'Motif':<20s} {'S\u00fbr %':>10s} {'Phishing %':>12s}")
print("\u2500" * 44)
for feat, name in zip(features, feature_names):
    safe_pct = 100 * df[df["label"]==0][feat].mean()
    phish_pct = 100 * df[df["label"]==1][feat].mean()
    print(f"{name:<20s} {safe_pct:>9.1f}% {phish_pct:>11.1f}%")

In [ ]:
fig, axes = plt.subplots(1, len(features), figsize=(16, 4))

for ax, feat, name in zip(axes, features, feature_names):
    safe_pct = 100 * df[df["label"]==0][feat].mean()
    phish_pct = 100 * df[df["label"]==1][feat].mean()
    bars = ax.bar(["S\u00fbr", "Phishing"], [safe_pct, phish_pct],
                  color=[COLORS["safe"], COLORS["phishing"]],
                  edgecolor="white", linewidth=1.5)
    ax.set_title(name, fontsize=12)
    ax.set_ylabel("% des emails")
    ax.set_ylim(0, 100)
    for bar, pct in zip(bars, [safe_pct, phish_pct]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f"{pct:.0f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

### Question pour toi

- Quel motif est le plus fort indicateur de phishing ?
- Est-ce que l'un de ces motifs pourrait provoquer des **fausses alertes** sur des emails sûrs ? Donne un exemple.

---

## 6 — Inspection des échantillons suspects

Les jeux de données du monde réel sont imparfaits. Certains emails peuvent être :
- **Mal étiquetés** (un email sûr marqué comme phishing, ou l'inverse)
- **Ambiguës** (pourraient être l'un ou l'autre)
- **Bruités** (texte corrompu, très court, ou non pertinent)

Regardons quelques échantillons potentiellement problématiques.

### Emails très courts

Les emails avec très peu de mots pourraient manquer d'informations suffisantes pour la classification.

In [ ]:
short_emails = df.nsmallest(10, "num_words")[["text", "label", "num_words"]]

print("Les 10 emails les plus courts du jeu de donn\u00e9es :\n")
for _, row in short_emails.iterrows():
    tag = "PHISHING" if row["label"] == 1 else "S\u00dbR"
    text_preview = row["text"][:120].replace("\n", " ")
    print(f"  [{tag:8s}] ({row['num_words']:3d} mots)  {text_preview}")

### Emails très longs

Les emails extrêmement longs peuvent contenir beaucoup de texte générique
(boilerplate) ou être des messages multi-parties avec des en-têtes.

In [ ]:
long_emails = df.nlargest(5, "num_words")[["text", "label", "num_words"]]

print("Les 5 emails les plus longs du jeu de donn\u00e9es :\n")
for _, row in long_emails.iterrows():
    tag = "PHISHING" if row["label"] == 1 else "S\u00dbR"
    text_preview = row["text"][:150].replace("\n", " ")
    print(f"  [{tag:8s}] ({row['num_words']:,} mots)  {text_preview}...")

### Potentiellement mal étiquetés : emails sûrs avec beaucoup de mots-clés de phishing

In [ ]:
import textwrap

phishing_keywords = ["urgent", "verify", "suspended", "click here",
                     "confirm your", "password", "act now", "limited time"]

def count_phishing_keywords(text):
    text_lower = text.lower()
    return sum(1 for kw in phishing_keywords if kw in text_lower)

df["phishing_keyword_count"] = df["text"].apply(count_phishing_keywords)

suspicious_safe = (df[(df["label"] == 0) & (df["phishing_keyword_count"] >= 2)]
                   .nlargest(3, "phishing_keyword_count"))

print("Emails s\u00fbrs qui contiennent beaucoup de mots-cl\u00e9s de type phishing :\n")
print("(Ils pourraient \u00eatre mal \u00e9tiquet\u00e9s, ou ce sont des emails l\u00e9gitimes")
print(" qui utilisent simplement un langage similaire.)\n")

for _, row in suspicious_safe.iterrows():
    print(f"{'='*70}")
    print(f"  Label : S\u00dbR  |  Mots-cl\u00e9s phishing trouv\u00e9s : {row['phishing_keyword_count']}")
    print(f"{'='*70}")
    print(textwrap.fill(row["text"][:400], width=70))
    print()

### Question pour toi

- Penses-tu que certains des emails ci-dessus sont mal étiquetés ? Pourquoi ?
- Pourquoi est-il important d'être conscient des données bruitées ou mal étiquetées
  avant d'entraîner un modèle ?

---

## 7 — Tableau récapitulatif

Rassemblons toutes nos découvertes dans un dernier résumé.

In [ ]:
n_safe = len(df[df["label"] == 0])
n_phish = len(df[df["label"] == 1])

print("\u2554\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2557")
print("\u2551       R\u00c9SUM\u00c9 DE L'EXPLORATION DU DATASET         \u2551")
print("\u2560\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2563")
print(f"\u2551  Total emails          : {len(df):>6,}                    \u2551")
print(f"\u2551  Emails s\u00fbrs           : {n_safe:>6,}  ({100*n_safe/len(df):.1f}%)           \u2551")
print(f"\u2551  Emails phishing       : {n_phish:>6,}  ({100*n_phish/len(df):.1f}%)           \u2551")
print(f"\u2551                                                      \u2551")
print(f"\u2551  Moy. mots (s\u00fbr)      : {df[df['label']==0]['num_words'].mean():>6.0f}                    \u2551")
print(f"\u2551  Moy. mots (phishing)  : {df[df['label']==1]['num_words'].mean():>6.0f}                    \u2551")
print(f"\u2551                                                      \u2551")
print(f"\u2551  Emails avec URL :                                   \u2551")
print(f"\u2551    S\u00fbr      : {100*df[df['label']==0]['has_url'].mean():>5.1f}%                            \u2551")
print(f"\u2551    Phishing : {100*df[df['label']==1]['has_url'].mean():>5.1f}%                            \u2551")
print("\u255a\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u255d")

---

## 8 — Qu'avons-nous appris ?

Écris quelques phrases résumant tes découvertes :

1. **Équilibre des classes :** Le jeu de données est-il équilibré ? Qu'est-ce que cela implique pour notre modèle ?

2. **Longueur des emails :** Les emails de phishing sont-ils plus courts, plus longs, ou à peu près pareils ?

3. **Mots clés :** Quels mots sont les meilleurs indicateurs de phishing ? D'emails sûrs ?

4. **Motifs :** Quels motifs (URLs, urgence, argent, \"click here\") sont les plus
   utiles pour la détection ?

5. **Qualité des données :** As-tu trouvé des emails bruités ou potentiellement mal étiquetés ?

*Écris ton résumé ici :*

1. ...
2. ...
3. ...
4. ...
5. ...

---

## Résumé

Dans ce notebook, tu as :

- Vérifié l'**équilibre des classes** et compris pourquoi c'est important.
- Comparé les **longueurs d'emails** entre emails sûrs et emails de phishing.
- Appris à **nettoyer du texte** (tokenisation, suppression des stop words).
- Trouvé les **mots les plus distinctifs** pour chaque classe.
- Détecté des **motifs spécifiques** (URLs, urgence, références à l'argent).
- Inspecté des **échantillons bruités et potentiellement mal étiquetés**.

Toutes ces observations nous aideront à comprendre ce que notre modèle
de machine learning va essayer d'apprendre dans le prochain notebook.

**À suivre :** Dans le *Notebook 02*, nous transformerons ces caractéristiques
textuelles en nombres grâce au **TF-IDF** et nous entraînerons nos premiers
classifieurs !